In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
print(df.head())

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
# Converting to datetime
df['datetime'] = pd.to_datetime(df['datetime'])

# Extracting the original features
df['hour'] = df['datetime'].dt.hour
df['day'] = df['datetime'].dt.day
df['month'] = df['datetime'].dt.month

# --- NEW FEATURE ENGINEERING ---
# Extracting Day of the Week (Monday=0, Sunday=6)
df['day_of_week'] = df['datetime'].dt.dayofweek

# Flagging the weekends (1 if Saturday/Sunday, 0 if Weekday)
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
# -------------------------------

print(df.head())

In [ ]:
print(df.isnull().sum())

df.drop_duplicates(inplace=True)

In [ ]:
hourly_usage = df.groupby('hour')['electricity_consumption'].mean()
monthly_usage = df.groupby('month')['electricity_consumption'].sum()
category_usage = df.groupby('var2')['electricity_consumption'].mean()

print(hourly_usage.head())

In [ ]:
df.to_csv("cleaned_data.csv", index=False)
hourly_usage.to_csv("hourly_usage.csv")
monthly_usage.to_csv("monthly_usage.csv")
category_usage.to_csv("category_usage.csv")

print("Data saved successfully!")

In [ ]:
print("Peak Hour:", hourly_usage.idxmax())
print("Highest Consumption Month:", monthly_usage.idxmax())
print("Top Category:", category_usage.idxmax())

In [ ]:
import matplotlib.pyplot as plt

hourly_usage.plot(kind='bar', title='Hourly Electricity Usage')
plt.xlabel("Hour")
plt.ylabel("Consumption")
plt.show()

In [ ]:
monthly_usage.plot(kind='line', title='Monthly Electricity Usage')
plt.xlabel("Month")
plt.ylabel("Consumption")
plt.show()

In [ ]:
category_usage.plot(kind='bar', title='Category-wise Usage')
plt.show()

In [ ]:
### Insights

##- Electricity consumption varies across different hours of the day.
##- Certain months show higher energy usage.
##- Some categories (var2) consume more electricity.

##This analysis helps improve transparency and encourages energy conservation.

In [ ]:
print("Final Data Shape:", df.shape)
print("\nAny Missing Values?")
print(df.isnull().sum())

In [ ]:
print(df.dtypes)

In [ ]:
print(df['electricity_consumption'].describe())

In [ ]:
### Data Handling Summary

##- Converted datetime and extracted features (hour, day, month)
##- Removed duplicates and handled missing values
##- Created aggregated datasets for analysis
##- Validated data for consistency and accuracy

##This ensures reliable data for visualization and modeling.

In [ ]:
print("Final shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())

In [ ]:
print(df['hour'].unique())
print(df['month'].unique())

In [ ]:
### Data Handling Summary

##- Loaded raw dataset
##- Cleaned and removed duplicates
##- Extracted time-based features (hour, day, month)
##- Created aggregated datasets for analysis
##- Validated data for consistency

##Prepared data for visualization and modeling.

In [ ]:
df.columns = df.columns.str.lower()

In [ ]:
### Apartment Data Transformation

##The dataset is adapted to simulate apartment-level electricity usage by introducing apartment identifiers and mapping categories to apartment types.

In [ ]:
df.rename(columns={'var2': 'apartment_type'}, inplace=True)

In [ ]:
df['apartment_id'] = 'A' + (df.index % 50).astype(str)

In [ ]:
df = df[['apartment_id', 'apartment_type', 'datetime', 
         'temperature', 'pressure', 'windspeed', 'var1',
         'electricity_consumption', 'hour', 'day', 'month', 'day_of_week', 'is_weekend']]


In [ ]:
apartment_usage = df.groupby('apartment_id')['electricity_consumption'].sum()
type_usage = df.groupby('apartment_type')['electricity_consumption'].mean()

In [ ]:
# Show output
print("Apartment Usage:")
print(apartment_usage.head())

print("\nType Usage:")
print(type_usage.head())

# Save files (keep this)
df.to_csv("apartment_energy_data.csv", index=False)
apartment_usage.to_csv("apartment_usage.csv")
type_usage.to_csv("type_usage.csv")

In [ ]:
import matplotlib.pyplot as plt

apartment_usage.head(10).plot(kind='bar')
plt.title("Electricity Usage per Apartment")
plt.xlabel("Apartment ID")
plt.ylabel("Consumption")
plt.show()

In [ ]:
type_usage.plot(kind='bar')
plt.title("Average Consumption by Apartment Type")
plt.xlabel("Apartment Type")
plt.ylabel("Average Consumption")
plt.show()

In [ ]:
apartment_usage.reset_index().to_csv("apartment_usage_clean.csv", index=False)
type_usage.reset_index().to_csv("type_usage_clean.csv", index=False)

In [ ]:
# Save the final cleaned version for the next sprint weeks
df.to_csv('cleaned_apartment_data.csv', index=False)
print("Week 2: Cleaned dataset is ready and saved.")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set the visual style
sns.set_theme(style="whitegrid")

# Create a boxplot to compare electricity consumption by apartment type
plt.figure(figsize=(10, 6))
sns.boxplot(x='apartment_type', y='electricity_consumption', data=df, palette='Set2')

# Add professional titles and labels for your demo
plt.title('Energy Consumption Distribution by Apartment Type', fontsize=15)
plt.xlabel('Apartment Type (Category)', fontsize=12)
plt.ylabel('Electricity Consumption (Units)', fontsize=12)

# Save the chart for your website framework
plt.savefig('consumption_boxplot.png')
plt.show()

In [ ]:
# Group by hour to see the building-wide trend
hourly_trend = df.groupby('hour')['electricity_consumption'].mean()

plt.figure(figsize=(12, 6))
hourly_trend.plot(kind='line', marker='o', color='b', linewidth=2)

# Highlighting the peak hour for the demo
plt.axvline(x=1, color='r', linestyle='--', label='Peak Usage (Hour 1)')

plt.title('Average Building Energy Consumption by Hour', fontsize=15)
plt.xlabel('Hour of the Day (24h Format)', fontsize=12)
plt.ylabel('Average Consumption (Units)', fontsize=12)
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='temperature', y='electricity_consumption', hue='apartment_type', data=df, alpha=0.5)

plt.title('Temperature vs. Electricity Consumption', fontsize=15)
plt.xlabel('Temperature', fontsize=12)
plt.ylabel('Consumption', fontsize=12)
plt.show()

In [ ]:
# Final Export for the Project Record
df.to_csv('final_energy_model_results.csv', index=False)
print("Project Complete: Final dataset exported.")

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None

print("Starting advanced ML pipeline...")

# 1. Advanced Feature Engineering
df['datetime'] = pd.to_datetime(df['datetime'])
df['day_of_year'] = df['datetime'].dt.dayofyear
df['quarter'] = df['datetime'].dt.quarter
df['is_month_start'] = df['datetime'].dt.is_month_start.astype(int)
df['is_month_end'] = df['datetime'].dt.is_month_end.astype(int)

df = df.sort_values(by=['apartment_id', 'datetime'])

# 2. Define Target Variable
y = df['electricity_consumption']

# 3. Define Features
feature_cols = ['hour', 'day', 'month', 'temperature', 'pressure', 'windspeed', 'var1', 
                'apartment_type', 'day_of_week', 'is_weekend', 'day_of_year', 
                'quarter', 'is_month_start', 'is_month_end']
X = df[feature_cols]

X_encoded = pd.get_dummies(X, columns=['apartment_type'], drop_first=True)

# 4. Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.20, random_state=42)
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

# 5. Train High-Capacity Models
print("\n--- Training RandomForestRegressor ---")
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=25,
    max_features='sqrt',
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_r2 = r2_score(y_test, rf_preds)

print("\n--- Training GradientBoostingRegressor ---")
gb_model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    random_state=42
)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)
gb_r2 = r2_score(y_test, gb_preds)

best_model = gb_model if gb_r2 > rf_r2 else rf_model
best_r2 = max(gb_r2, rf_r2)
best_name = 'Gradient Boosting' if gb_r2 > rf_r2 else 'Random Forest'
xgb_r2 = None

if XGBRegressor:
    print("\n--- Training XGBRegressor ---")
    xgb_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    xgb_model.fit(X_train, y_train)
    xgb_preds = xgb_model.predict(X_test)
    xgb_r2 = r2_score(y_test, xgb_preds)
    
    if xgb_r2 > best_r2:
        best_model = xgb_model
        best_r2 = xgb_r2
        best_name = 'XGBoost'

# --- FINAL COMPARISON PRINTOUT ---
print("\n" + "="*40)
print("    FINAL ML MODEL ACCURACY RESULTS")
print("="*40)
print(f"1. Random Forest:      {rf_r2:.4f}")
print(f"2. Gradient Boosting:  {gb_r2:.4f}")
if xgb_r2 is not None:
    print(f"3. XGBoost:            {xgb_r2:.4f}")
else:
    print("3. XGBoost:            [Not Installed]")
print("-"*40)
print(f"WINNER: {best_name} ({best_r2:.4f})")
print("="*40 + "\n")

# 6. Save the best model
with open('final_energy_model_v2.pkl', 'wb') as file:
    pickle.dump(best_model, file)
print(f"Best model ({best_name}) saved as 'final_energy_model_v2.pkl'")


In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

print("Starting advanced ML pipeline...")

# 1. Advanced Feature Engineering
df['datetime'] = pd.to_datetime(df['datetime'])
df['day_of_year'] = df['datetime'].dt.dayofyear
df['quarter'] = df['datetime'].dt.quarter
df['is_month_start'] = df['datetime'].dt.is_month_start.astype(int)
df['is_month_end'] = df['datetime'].dt.is_month_end.astype(int)

# Cyclical Encoding for Time Features (Massive boost for Linear Regression)
df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
df['month_sin'] = np.sin(2 * np.pi * df['month']/12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month']/12.0)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week']/7.0)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week']/7.0)

df = df.sort_values(by=['apartment_id', 'datetime'])

# 2. Define Target Variable
y = df['electricity_consumption']

# 3. Define Features
feature_cols = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 
                'temperature', 'pressure', 'windspeed', 'var1', 
                'apartment_type', 'is_weekend', 'day_of_year', 
                'quarter', 'is_month_start', 'is_month_end']
X = df[feature_cols]

X_encoded = pd.get_dummies(X, columns=['apartment_type'], drop_first=True)

# Scale the features (Required to maximize Linear Regression)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_encoded), columns=X_encoded.columns)

# 4. Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.20, random_state=42)
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

# 5. Train the 3 Required Models

print("\n--- Training Linear Regression ---")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_r2 = r2_score(y_test, lr_preds)

print("\n--- Training Decision Tree ---")
dt_model = DecisionTreeRegressor(random_state=42, max_depth=20, min_samples_leaf=4, min_samples_split=10)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
dt_r2 = r2_score(y_test, dt_preds)

print("\n--- Training Gradient Boosting ---")
gb_model = GradientBoostingRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    max_features='sqrt',
    random_state=42
)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)
gb_r2 = r2_score(y_test, gb_preds)

# Determine Winner
models = [('Linear Regression', lr_model, lr_r2), 
          ('Decision Tree', dt_model, dt_r2), 
          ('Gradient Boosting', gb_model, gb_r2)]
best_name, best_model, best_r2 = max(models, key=lambda item: item[2])

# --- FINAL COMPARISON PRINTOUT ---
print("\n" + "="*40)
print("    FINAL ML MODEL ACCURACY RESULTS")
print("="*40)
print(f"1. Linear Regression:  {lr_r2:.4f}")
print(f"2. Decision Tree:      {dt_r2:.4f}")
print(f"3. Gradient Boosting:  {gb_r2:.4f}")
print("-"*40)
print(f"WINNER: {best_name} ({best_r2:.4f})")
print("="*40 + "\n")

# 6. Save the best model
with open('final_energy_model_v2.pkl', 'wb') as file:
    pickle.dump(best_model, file)
print(f"Best model ({best_name}) saved as 'final_energy_model_v2.pkl'")
